In [1]:
from langchain_core.tools import tool
import ast, operator

# [1번 문제]
# 도시 이름을 받아 날씨를 반환하는 get_weather(city: str) 함수를
# @tool 데코레이터와 함께 작성하세요. (docstring 필수!)
@tool
def get_weather(city: str) -> str:
    """도시 이름을 받아 날씨 정보를 반환합니다."""
    weather_db = {"서울": "맑음, 23도"}
    return weather_db.get(city, "정보 없음")

# [2번 문제]
# 사칙연산 문자열(예: "3 + 5")을 계산해 반환하는
# calculate(expression: str) 함수를 @tool로 작성하세요.
# (ast.parse와 ast.BinOp를 활용하면 eval보다 안전하게 계산할 수 있습니다)
# eval("5+6") --> 파이썬 코드로 실행해줌
_OPS = {
    ast.Add: operator.add,      # + 기호
    ast.Sub: operator.sub,      # - 기호
    ast.Mult: operator.mul,     # * 기호
    ast.Div: operator.truediv,  # / 기호
    ast.USub: operator.neg,     # 음수 부호 (-)
    ast.UAdd: operator.pos      # 양수 부호 (+)
}
def _safe_eval(node): # 함수명 앞에 _를 붙인건 private의 의미임
    if isinstance(node, ast.Constant):# 숫자인지?
        return node.value
    if isinstance(node, ast.BinOp):# Binary Operation 이항연산자인가?
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp):# 단항연산자인가?
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("허용되지 않은 수식입니다.")

@tool
def calculate(expression: str) -> str:
    """사칙연산 수식 문자열을 계산합니다. 예: '3 + 5 * 2'"""
    try:# mode를 eval로 지정하면 expression문자열에 단일 수식만 허용하고 
        tree = ast.parse(expression, mode="eval") # 결과물은 Tree 객체로 반환(연산자 우선순위로 정리된 계층구조)
        result = _safe_eval(tree.body) # Tree 안에 값을 꺼내서 함수로 전달
        return f"{expression} = {result}"
    except Exception:
        return "계산할 수 없는 수식입니다."
# [도전 문제]
# get_weather.invoke({"city": "부산"}) 와
# calculate.invoke({"expression": "10 * (2 + 3)"}) 를 각각 실행해
# Tool 함수가 Agent 없이도 단독으로 잘 동작하는지 먼저 확인하세요.


# [도전 문제]
# 1. 날씨 조회 툴 테스트
result_weather = get_weather.invoke({"city": "부산"})
print("날씨 결과:", result_weather)

# 2. 계산기 툴 테스트
result_calc = calculate.invoke({"expression": "10 * (2 + 3)"})
print("계산 결과:", result_calc)



날씨 결과: 정보 없음
계산 결과: 10 * (2 + 3) = 50


In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# 1. 환경변수(.env) 로드 및 LLM 모델 생성 (이 부분이 빠져 있었습니다!)
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. 에이전트 생성 (작성하신 코드 그대로!)
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="너는 도움이 되는 비서야.",
)

# 3. 에이전트 호출
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "서울 날씨 어때?"}
    ]
})


# result = {
#     "messages": [
#         HumanMessage(content="서울 날씨 어때?"),       # [0] 사용자의 질문
#         AIMessage(content="", tool_calls=[...]),      # [1] AI: "날씨 툴 실행해줘"
#         ToolMessage(content="맑음, 23도"),             # [2] 도구가 알려준 날씨 결과
#         AIMessage(content="서울은 현재 맑고 23도입니다.") # [3] AI의 최종 답변!
#     ]
# }


# 4. 최종 응답 출력
print(result["messages"][1].tool_calls)
final = result["messages"][-1]
print(final.content)




[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_316cyCivFqNvKBoFmHOVPRS6', 'type': 'tool_call'}]
서울의 날씨는 맑고 기온은 23도입니다.


In [3]:
from langchain.agents import create_agent

# [1번 문제]
# 앞 슬라이드에서 만든 get_weather 하나만 tools로 갖는 agent를
# create_agent(model=llm, tools=[...], system_prompt="...") 로 만드세요.

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="너는 도움이 되는 비서야.",
)

# [2번 문제]
# agent.invoke({"messages": [{"role": "user", "content": "대구 날씨 어때?"}]})
# 를 실행하고, result["messages"][-1].content를 출력하세요.

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "대구 날씨 어때?"}
    ]
})

final = result["messages"][-1]
print(final.content)

# [도전 문제]
# result["messages"] 를 순서대로 출력해보며, HumanMessage ->
# AIMessage(tool_calls) -> ToolMessage -> AIMessage(최종 답변) 흐름이
# 실제로 몇 개의 메시지로 이루어지는지 세어보세요.
# [도전 문제]
# 1. 전체 메시지 개수 출력
print(f"총 메시지 개수: {len(result['messages'])}개\n")

# 2. 순서대로 각 메시지의 종류와 내용 출력
for i, msg in enumerate(result["messages"], 1):
    print(f"[{i}단계] 타입: {type(msg).__name__}")
    print(f"내용: {msg.content}")
    
    # AI가 도구를 호출한 경우(AIMessage에 tool_calls가 있는 경우) 추가 확인
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"-> 호출한 도구 정보: {msg.tool_calls}")
        
    print("-" * 50)





대구의 날씨 정보를 찾을 수 없습니다. 다른 도시의 날씨를 알고 싶으신가요?
총 메시지 개수: 4개

[1단계] 타입: HumanMessage
내용: 대구 날씨 어때?
--------------------------------------------------
[2단계] 타입: AIMessage
내용: 
-> 호출한 도구 정보: [{'name': 'get_weather', 'args': {'city': '대구'}, 'id': 'call_0sNZ2YE5bcmAhx5tmhbYt0ZY', 'type': 'tool_call'}]
--------------------------------------------------
[3단계] 타입: ToolMessage
내용: 정보 없음
--------------------------------------------------
[4단계] 타입: AIMessage
내용: 대구의 날씨 정보를 찾을 수 없습니다. 다른 도시의 날씨를 알고 싶으신가요?
--------------------------------------------------


In [9]:
# 앞 슬라이드의 get_weather, calculate 두 Tool을 모두 사용합니다.

# [1번 문제]
# [get_weather, calculate] 두 Tool을 가진 agent를 만드세요.
agent = create_agent(
    model=llm,
    tools=[get_weather, calculate],
    system_prompt="필요할 때만 도구를 써.",
)


# [2번 문제]
# "대구 날씨 알려줘" 와 "7 곱하기 8은 얼마야?" 두 질문을 각각
# invoke()해서, 서로 다른 도구가 호출되는지 답변으로 확인하세요.
results = agent.invoke({"messages": [
    {"role": "user", "content": "대구 날씨 알려줘"}]})

print(results["messages"][-1].content)


# [도전 문제]
# "안녕하세요!" 처럼 두 도구 모두와 관련 없는 질문도 invoke()해보고,
# result["messages"] 안에 ToolMessage가 있는지 없는지 확인해
# Agent가 도구를 쓰지 않고 바로 답했는지 판단해보세요.

# 여기에 코드를 작성해보세요


results = agent.invoke({"messages": [
    {"role": "user", "content": "안녕하세요"}]})

print(results["messages"][1])
print(results["messages"][-1].content)


대구의 날씨 정보를 찾을 수 없습니다. 다른 도시의 날씨를 알고 싶으신가요?
content='안녕하세요! 어떻게 도와드릴까요?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 95, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_109d01f5b1', 'id': 'chatcmpl-Ed6b7nTjoYLYkPFTGIKTqpFjwjlQS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a17676-cae4-7135-a753-b53cafca13dd-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 95, 'output_tokens': 11, 'total_tokens': 106, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning

In [10]:
import os
from dotenv import load_dotenv

# [1. 필요한 부품(라이브러리) 가져오기]
from langchain_openai import ChatOpenAI        # 두뇌 역할을 할 ChatGPT 모델
from langchain_openai import OpenAIEmbeddings   # 글자를 컴퓨터가 계산할 수 있는 '숫자(좌표)'로 바꿔주는 번역기
from langchain_core.vectorstores import InMemoryVectorStore  # [수정: 추가] 컴퓨터 메모리에 문서를 저장해두는 '미니 도서관'
from langchain_core.tools import tool          # [수정: 추가] 일반 파이썬 함수를 AI가 쓸 수 있는 '도구'로 만들어주는 마법 도장
from langchain.agents import create_agent      # 도구를 쥐어주고 에이전트(인공지능 비서)를 만드는 공장


# [2. 비밀번호(API 키) 준비]
# 컴퓨터 속 .env 파일에 적어둔 OPENAI_API_KEY를 읽어옵니다.
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")


# [3. AI 두뇌(LLM) 만들기]
# 질문에 답변을 생성할 모델을 준비합니다.
llm = ChatOpenAI(
    model="gpt-4o-mini",  # 빠르고 똑똑한 GPT-4o-mini 모델 선택
    api_key=api_key
)


# [4. 글자 번역기(임베딩) 만들기]
# 문장의 '의미'를 숫자로 바꿔서, 나중에 질문과 내용이 비슷한 문장을 찾을 수 있게 해줍니다.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


# [5. 공지사항 문서 조각들(청크) 준비]
# 긴 글을 한 번에 읽기 힘드니 책의 페이지처럼 작게 쪼개놓은 문서 조각들입니다.
notice_chunks = [
    "[공지] 2026년 11월 1일(일) 02:00부터 06:00까지 시스템 안정화를 위한 정기 서버 점검이 진행됩니다.",
    "[영향도] 점검 시간 동안 로그인, 결제 서비스, 마이페이지 등 주요 기능의 이용이 일시 중단됩니다.",
    "[주의사항] 데이터 유실 방지를 위해 점검 시작 전 진행 중인 작업 및 결제는 미리 완료/저장해 주시기 바랍니다.",
    "[문의] 점검 완료 후 즉시 정상화될 예정이며, 관련 문의는 고객센터(1588-0000) 또는 1:1 문의를 이용해 주세요."
]


# [6. 미니 도서관(벡터 저장소)에 책 꽂아두기]
# [수정: noitce_chunks -> notice_chunks 오타 수정]
# 글자들을 임베딩 번역기로 숫자로 바꾼 뒤, 검색하기 좋게 메모리에 저장합니다.
vectorstore = InMemoryVectorStore.from_texts(
    texts=notice_chunks,
    embedding=embeddings
)


# [7. 도서관 사서(검색기, Retriever) 고용하기]
# 도서관(vectorstore)에서 질문과 가장 관련 있는 조각 2개(k: 2)만 쏙 골라오는 역할을 합니다.
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)


# [8. AI 비서가 손에 쥘 '도구(Tool)' 만들기]
# @tool을 붙여주면 AI가 "아, 내가 스스로 필요할 때 이 함수를 실행할 수 있구나!"라고 인식합니다.
@tool
def search_notice(query: str) -> str:
    """공지사항에서 관련 내용을 검색합니다."""
    # ↑ 이 주석(설명)이 매우 중요합니다! AI는 이 설명을 읽고 "언제 이 도구를 써야 할지" 스스로 결정합니다.
    
    # 사서(retriever)에게 질문(query)을 주고 관련 있는 문서 2장을 찾아오라고 시킵니다.
    docs = retriever.invoke(query)
    
    # 찾아온 문서 2장의 글 내용(page_content)을 줄바꿈(\n)으로 합쳐서 AI에게 건네줍니다.
    return "\n".join(d.page_content for d in docs)


In [12]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from langchain.agents import create_agent

docs = [
    "환불 정책: 강의 시작 후 7일 이내에는 전액 환불이 가능합니다.",
    "수료증 발급: 전체 강의의 80% 이상을 수강하면 신청할 수 있습니다.",
]

# ==========================================================
# [1번 문제]
# docs로 vectorstore와 retriever(k=2)를 만들고, 
# 검색 결과를 하나의 문자열로 반환하는 search_notice(query: str) 함수를 @tool로 작성하세요.
# ==========================================================

# 1) docs 목록으로 벡터 저장소 만들기
vectorstore = InMemoryVectorStore.from_texts(texts=docs, embedding=embeddings)

# 2) 관련 문서 2개를 찾아오는 retriever 만들기
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 3) 검색 결과를 하나의 문자열로 합쳐 반환하는 @tool 함수 만들기
@tool
def search_notice(query: str) -> str:
    """공지사항 및 규정(환불 정책, 수료증 등) 관련 내용을 검색합니다."""
    retrieved_docs = retriever.invoke(query)
    return "\n".join(d.page_content for d in retrieved_docs)


# ==========================================================
# [2번 문제]
# tools=[search_notice] 로 agent를 만들고, 
# "환불 규정이 어떻게 되나요?" 로 invoke()해 답변을 확인하세요.
# ==========================================================

# 1) search_notice 도구를 가진 agent 생성
agent = create_agent(
    model=llm,
    tools=[search_notice],
    system_prompt="너는 고객 지원 비서야. 규정이나 공지사항 관련 질문에는 도구를 검색해서 정확하게 답변해줘.",
)

# 2) 질문 호출
result = agent.invoke({
    "messages": [{"role": "user", "content": "환불 규정이 어떻게 되나요?"}]
})

# 3) 최종 답변 출력
print("[최종 답변]")
print(result["messages"][-1].content)
print("-" * 50)


# ==========================================================
# [도전 문제]
# result["messages"] 에서 ToolMessage의 내용을 출력해, 
# 실제로 어떤 문서가 검색되어 답변에 활용됐는지 확인하세요.
# ==========================================================

for msg in result["messages"]:
    # 메시지들 중에서 ToolMessage 타입인 것만 골라냅니다
    if isinstance(msg, ToolMessage):
        print("[실제 검색되어 활용된 문서 (ToolMessage)]")
        print(msg.content)


[최종 답변]
환불 규정은 다음과 같습니다: 

- 강의 시작 후 7일 이내에는 전액 환불이 가능합니다. 

추가로, 수료증 발급은 전체 강의의 80% 이상을 수강해야 신청할 수 있습니다.
--------------------------------------------------
[실제 검색되어 활용된 문서 (ToolMessage)]
환불 정책: 강의 시작 후 7일 이내에는 전액 환불이 가능합니다.
수료증 발급: 전체 강의의 80% 이상을 수강하면 신청할 수 있습니다.


In [13]:
from langchain.agents import create_agent

# ==========================================================
# [1번 문제] 
# tools=[get_weather, calculate, search_notice] 로 agent를 만들고, 
# system_prompt에 '여러 도구를 상황에 맞게 사용하는 비서' 역할을 적으세요.
# ==========================================================

agent = create_agent(
    model=llm,
    tools=[get_weather, calculate, search_notice],
    system_prompt="너는 여러 도구를 상황에 맞게 사용하는 똑똑한 비서야. 질문의 성격에 맞는 도구를 골라 쓰고, 도구가 필요 없는 일상 대화는 직접 답변해줘."
)


# ==========================================================
# [2번 문제] 
# question(문자열)을 받아 agent.invoke() 결과의 최종 답변을 
# 반환하는 ask_agent(question) 함수를 작성하세요.
# ==========================================================

def ask_agent(question: str) -> str:
    """질문을 받아 에이전트에게 묻고, 최종 답변 텍스트만 쏙 꺼내서 돌려주는 함수"""
    result = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    # messages 리스트의 맨 마지막(-1)에 AI의 최종 답변이 들어있습니다.
    return result["messages"][-1].content


# ==========================================================
# [도전 문제] 
# 4가지 질문을 각각 ask_agent()로 실행해보고, 
# 세 도구와 '도구 없이 바로 답하기'가 상황에 맞게 쓰이는지 확인하세요.
# ==========================================================

questions = [
    "제주 날씨 어때?",            # -> get_weather 도구 사용
    "125 더하기 375는?",          # -> calculate 도구 사용
    "수료증은 어떻게 받나요?",     # -> search_notice 도구 사용
    "오늘 기분이 좋아요!"         # -> 도구 없이 모델이 바로 답변
]

for q in questions:
    print(f"Q: {q}")
    answer = ask_agent(q)
    print(f"A: {answer}")
    print("-" * 50)


Q: 제주 날씨 어때?
A: 현재 제주 날씨에 대한 정보가 제공되지 않네요. 다른 질문이나 필요하신 정보가 있다면 말씀해 주세요!
--------------------------------------------------
Q: 125 더하기 375는?
A: 125 더하기 375는 500입니다.
--------------------------------------------------
Q: 수료증은 어떻게 받나요?
A: 수료증은 전체 강의의 80% 이상을 수강하면 신청할 수 있습니다. 추가로, 환불 정책은 강의 시작 후 7일 이내에는 전액 환불이 가능합니다. 다른 질문이 있다면 말씀해 주세요!
--------------------------------------------------
Q: 오늘 기분이 좋아요!
A: 정말 기쁜 소식이네요! 좋은 기분을 유지하시길 바랍니다. 어떤 특별한 일이 있었나요?
--------------------------------------------------


In [14]:
from langchain_core.messages import ToolMessage

def ask_agent_with_check(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    
    # 메시지 중에 ToolMessage가 있는지 확인
    tools_used = [msg.name for msg in result["messages"] if isinstance(msg, ToolMessage)]
    
    if tools_used:
        print(f"  ↳ [사용한 도구]: {', '.join(tools_used)}")
    else:
        print("  ↳ [도구 사용 안 함]: AI가 직접 답변")
        
    print(f"  ↳ [최종 답변]: {result['messages'][-1].content}")

# 확인해보기
for q in questions:
    print(f"질문: {q}")
    ask_agent_with_check(q)
    print()


질문: 제주 날씨 어때?
  ↳ [사용한 도구]: get_weather
  ↳ [최종 답변]: 죄송하지만 제주 날씨 정보를 확인할 수 없습니다. 다른 도움이 필요하시면 말씀해 주세요!

질문: 125 더하기 375는?
  ↳ [사용한 도구]: calculate
  ↳ [최종 답변]: 125 더하기 375는 500입니다.

질문: 수료증은 어떻게 받나요?
  ↳ [사용한 도구]: search_notice
  ↳ [최종 답변]: 수료증은 전체 강의의 80% 이상을 수강한 후에 신청할 수 있습니다. 추가적인 질문이 있으시면 알려주세요!

질문: 오늘 기분이 좋아요!
  ↳ [도구 사용 안 함]: AI가 직접 답변
  ↳ [최종 답변]: 좋은 기분이 드는 날이군요! 어떤 일이 있어서 그런가요? 😊

